# Step 2 — Prompts and output parsing

**Goal:** one set of prompt templates for every model (`prompts/v0/*.txt`, frozen as `v1` after the pilot)
and one parser that turns any model's output into `label` + `confidence`.

| Template | Task | Calls | Output |
|---|---|---|---|
| `faith_m1` | A | 1 | `{"reasoning", "label": SUPPORTED/UNSUPPORTED, "confidence"}` |
| `faith_m2_extract` → `faith_m2_verify` | A | 2 | claims, then one verdict per claim; **we** compute the label (any false → UNSUPPORTED) |
| `faith_m3` | A (optional) | 1 | M1 + 3 anchors from the **train** split (hand-picked in Step 6) |
| `pairwise_m1` | B | 1 per order | `{"reasoning", "label": A/B, "confidence"}`; every pair runs in both orders |

Runtime: CPU.

In [1]:
# --- Setup: mount Drive, go to the project, install, import ---------------------------
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LLM_as_Judge_Benchmark'
REPO_URL = 'https://github.com/moghbaie/LLM_as_Judge_Benchmark.git'

import os, sys, subprocess
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
!git pull -q || true
!pip -q install -r requirements.txt
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)

Mounted at /content/drive


## 1. Tests (41 cases: rendering, swap logic, 23 messy-output parsing cases)

In [2]:
!python -m pytest -q tests/test_step2_prompts.py

.........................................                                [100%]
41 passed in 2.21s


## 2. Look at a rendered prompt for each dataset

In [3]:
from ljbench import prompts
from ljbench.io_utils import read_jsonl
print('prompt version:', prompts.PROMPT_VERSION, '| template hashes:', prompts.template_hashes())
for name, build in [('ragtruth_test_600', prompts.faith_m1), ('faithbench_250', prompts.faith_m1),
                    ('judgebench_eval_600', prompts.pairwise_m1)]:
    item = read_jsonl(f'data/{name}.jsonl').to_dict(orient='records')[0]
    msgs = build(item)
    print(f'\n{"#"*20} {name} | {item["id"]} | gold = {item["human_label"]}')
    print(msgs[1]['content'][:2500] + ('\n...[truncated for display]' if len(msgs[1]['content']) > 2500 else ''))

prompt version: v0 | template hashes: {'faith_m1': '3bd648f3b6c8', 'faith_m2_extract': 'c2b752ed9b9b', 'faith_m2_verify': 'b4cca5373b2e', 'faith_m3': 'f31447f38231', 'pairwise_m1': 'b667766bbf37', 'system': 'd1ecc649c844'}

#################### ragtruth_test_600 | ragtruth-8077 | gold = SUPPORTED
Decide whether an AI-generated response is faithful to its source material.

- SUPPORTED: every factual claim in the response is stated in, or directly inferable from, the source material.
- UNSUPPORTED: at least one claim contradicts the source material, or adds information that cannot be verified from it (for example an added detail, a wrong number, name or date, or an inference presented as fact).

Judge only faithfulness to the source. Ignore writing quality and completeness, and do not use your own world knowledge.

<source>

{'name': 'Bogo SB', 'address': '1114 State St', 'city': 'Santa Barbara', 'state': 'CA', 'categories': 'Food, Food Delivery Services, Burgers, Pizza, Caterers, Event 

## 3. M2 prompts and the A/B swap

In [4]:
item = read_jsonl('data/faithbench_250.jsonl').to_dict(orient='records')[0]
print(prompts.faith_m2_extract(item)[1]['content'])
print('\n' + '='*80 + '\n')
print(prompts.faith_m2_verify(item, ['James Jones won the 1952 National Book Award.',
                                     'James Jones is also a basketball player.'])[1]['content'])
pair = read_jsonl('data/judgebench_dev_20.jsonl').to_dict(orient='records')[0]
sw = prompts.swap_pair(pair)
print('\noriginal gold:', pair['human_label'], '| swapped gold:', sw['human_label'],
      '| answers exchanged:', sw['answer'] == pair['answer_b'])

Break the response below into atomic factual claims.

- Each claim is one short, self-contained statement (replace pronouns with the names they refer to).
- Together, the claims must cover every factual statement in the response.
- Copy the meaning faithfully: do not add, correct or judge anything.
- Skip statements that only say information is unavailable, and skip greetings or remarks about the task.

<response>
James Jones is an author who won the 1952 National Book Award for his first published novel, "From Here to Eternity," which was adapted into a film and a TV series. He is also a basketball player who averaged 25 points per game as a senior, earning accolades and winning three NBA championships. Jones and LeBron James reached the NBA Finals for seven consecutive years in 2017, and Jones finished third in three-point percentage during the 2007-08 season and won the Three-Point Contest in 2011.
</response>

Reply with JSON only, in exactly this format:
{"claims": ["<claim 1>", "

## 4. The parser on realistic messy outputs

In [5]:
from ljbench.parsing import parse_judgement
examples = [
    '{"reasoning": "All claims appear in the source.", "label": "SUPPORTED", "confidence": 92}',
    '```json\n{"reasoning": "Wrong year.", "label": "UNSUPPORTED", "confidence": 85}\n```',
    'Let me check.\n<think>claim 3 adds a date...</think>\n{"reasoning":"Adds a date","label":"unsupported","confidence":0.7}',
    "{'reasoning': 'ok', 'label': 'SUPPORTED', 'confidence': 60,}",
    '{"reasoning": "The response is cut off here because max_tokens ran out, "label": "UNSUPPORTED"',
    'The response is faithful.',
    '{"reasoning": "x", "label": "PARTIALLY SUPPORTED", "confidence": 50}',
]
rows = []
for e in examples:
    p = parse_judgement(e, 'faithfulness')
    rows.append({'raw output': e[:70], 'label': p.label, 'confidence': p.confidence, 'mode': p.mode, 'error': p.error})
pd.DataFrame(rows)

,raw output,label,confidence,mode,error
0,"{""reasoning"": ""All claims appear in the source."", ""label"": ""SUPPORTED""",SUPPORTED,92.0,strict,None
1,"```json\n{""reasoning"": ""Wrong year."", ""label"": ""UNSUPPORTED"", ""confiden",UNSUPPORTED,85.0,fenced,None
2,"Let me check.\n<think>claim 3 adds a date...</think>\n{""reasoning"":""Adds",UNSUPPORTED,70.0,extracted,None
3,"{'reasoning': 'ok', 'label': 'SUPPORTED', 'confidence': 60,}",SUPPORTED,60.0,repaired,None
4,"{""reasoning"": ""The response is cut off here because max_tokens ran out",UNSUPPORTED,NaN,regex,None
5,The response is faithful.,None,NaN,failed,no JSON object with a label found
6,"{""reasoning"": ""x"", ""label"": ""PARTIALLY SUPPORTED"", ""confidence"": 50}",None,NaN,failed,invalid label 'PARTIALLY SUPPORTED'


## Result and explanation

Expected: **41 passed.**

- **Prompts are identical for every model.** Item text is inserted verbatim with `string.Template`, so `$` or `{}` in a
  source document can't break a prompt (tested with `"Price is $5 {not json}"`).
- **Template overhead** is about 270 tokens (M1), 180 + 200 (M2) and 220 (pairwise). With that, the core run is about 3.9M input tokens per model.
- **Parse modes**: `strict` (pure JSON) > `fenced` > `extracted` (JSON plus chatter) > `repaired` (trailing comma, single quotes) > `regex`.
  Lenient modes keep small models from being punished for formatting, while `parse_mode` is stored with each result, so we can report
  *how* clean each model's output was. Anything without a valid label is `None`, which **counts as wrong**.
- **M2's label is computed by us, not the model:** any unsupported or missing claim verdict means UNSUPPORTED. A response with zero claims counts as SUPPORTED.
- **M3 anchors** don't exist yet (`prompts/v0/fewshot_anchors.json`). They're hand-picked from the train pool in Step 6, and the code refuses test-split anchors.

**Gate passed → Step 3 (metrics).**